In [3]:
library(ggplot2)
library(behavr)
library(scopr)
library(sleepr)
library(ggetho)
library(plotly)
#library(survival)
library(cowplot)
#library(ggthemes)
library(plotly)
library(data.table)
library(stringi)
library(ggtern)
library(ggpubr)
library(EnvStats)
library(RColorBrewer)
library(dplyr)
library(plyr)

Loading required package: data.table


Attaching package: ‘plotly’


The following object is masked from ‘package:ggplot2’:

    last_plot


The following object is masked from ‘package:stats’:

    filter


The following object is masked from ‘package:graphics’:

    layout




ERROR: Error in library(ggtern): there is no package called ‘ggtern’


In [1]:
REMOTE_DATA_SOURCE <- "ftp://turing.lab.gilest.ro/auto_generated_data/ethoscope_results/"
MY_DIR <- "/home/hjones/"
setwd(MY_DIR)

# This is the placement of the data in this computer, make sure this is a file where you want it saved!
DATA_DIR <- "/mnt/ethoscope_results"
#This is the placement of the real data, on the NAS

#this is the place were a cache version of the data is stored, once it has been taken from NAS. This make the loading faster on the second time.
CACHE <- "/home/cache"

#This is the query of the experiment (this is the table you made of the data)
METADATA <- "/home/hjones/ethoscope_metadata_cs_r.csv"

In [4]:
#To get the files from the remote source
query <- link_ethoscope_metadata(METADATA,
                                 result_dir = DATA_DIR)

In [5]:
#This is the magic step, it loads the data to R and applies a function at the same time, in this case, the asleep annotation.
dt <- load_ethoscope(query,
                     reference_hour = 9.0, 
                     FUN = sleep_annotation,
                     velocity_correction_coef = 0.01,
                     cache = CACHE)

Loading ROI number 1 from:
	/mnt/ethoscope_results/10920c31fced4496b6d8c22165759469/ETHOSCOPE_109/2022-02-24_09-34-45/2022-02-24_09-34-45_10920c31fced4496b6d8c22165759469.db
Loading ROI number 2 from:
	/mnt/ethoscope_results/10920c31fced4496b6d8c22165759469/ETHOSCOPE_109/2022-02-24_09-34-45/2022-02-24_09-34-45_10920c31fced4496b6d8c22165759469.db
Loading ROI number 3 from:
	/mnt/ethoscope_results/10920c31fced4496b6d8c22165759469/ETHOSCOPE_109/2022-02-24_09-34-45/2022-02-24_09-34-45_10920c31fced4496b6d8c22165759469.db
Loading ROI number 4 from:
	/mnt/ethoscope_results/10920c31fced4496b6d8c22165759469/ETHOSCOPE_109/2022-02-24_09-34-45/2022-02-24_09-34-45_10920c31fced4496b6d8c22165759469.db
Loading ROI number 5 from:
	/mnt/ethoscope_results/10920c31fced4496b6d8c22165759469/ETHOSCOPE_109/2022-02-24_09-34-45/2022-02-24_09-34-45_10920c31fced4496b6d8c22165759469.db
Loading ROI number 6 from:
	/mnt/ethoscope_results/10920c31fced4496b6d8c22165759469/ETHOSCOPE_109/2022-02-24_09-34-45/2022-02-24_0

In [6]:
#to check whether there are dead animals.
dt_curated <- curate_dead_animals(dt)
summary(dt_curated)

behavr table with:
 168	individuals
 11	metavariables
 8	variables
 3.018158e+06	measurements
 1	key (id)


In [ ]:
#subset for 12 hours of time
dt_curated<- dt_curated[t>days(0) &t<days(0.5)]

In [8]:
#subset for compound
dt_CS_solvent <-dt_curated[xmv(compound)=="solvent",]
dt_CS_ddt <-dt_curated[xmv(compound)=="ddt", ]
dt_CS_dieldrin <- dt_curated[xmv(compound)=="dieldrin",]

In [10]:
#do this for each group - saves invidual csv files of max velocity in a folder of your choice
#with ascending numbers to denote each individual
dt_CS_solvent <- dt_CS_solvent[dt_CS_solvent, meta=T]
#make table with only max velocity data by id
dt_CS_solvent_velocity <- dt_CS_solvent[, .(max_velocity), by=id]
#quick summary to check the data
summary(dt_CS_solvent_velocity)
#puts the data into a list
split.df <- split(dt_CS_solvent_velocity, dt_CS_solvent_velocity$id)
for(i in 1:length(split.df)){
  write.csv(split.df[[i]], paste0("/home/hjones/CS/solvent/CS_control_",i,
                                  ".csv"))
}

ERROR: Error in `[.data.table`(dt_CS_solvent, dt_CS_solvent, meta = T): unused argument (meta = T)


In [ ]:
#do this for each group - saves invidual csv files of max velocity in a folder of your choice
#with ascending numbers to denote each individual
dt_CS_ddt <- dt_CS_ddt[dt_CS_ddt, meta=T]
#make table with only max velocity data by id
dt_CS_ddt_velocity <- dt_CS_ddt[, .(max_velocity), by=id]
#quick summary to check the data
summary(dt_CS_ddt_velocity)
#puts the data into a list
split.df <- split(dt_CS_ddt_velocity,dt_CS_ddt_velocity$id)
for(i in 1:length(split.df)){
  write.csv(split.df[[i]], paste0("/home/hjones/CS/ddt/CS_ddt_",i,
                                  ".csv"))
}

In [ ]:
#do this for each group - saves invidual csv files of max velocity in a folder of your choice
#with ascending numbers to denote each individual
dt_CS_dieldrin <- dt_CS_dieldrin[dt_CS_dieldrin, meta=T]
#make table with only max velocity data by id
dt_CS_dieldrin_velocity <- dt_CS_dieldrin[, .(max_velocity), by=id]
#quick summary to check the data
summary(dt_CS_dieldrin_velocity)
#puts the data into a list
split.df <- split(dt_CS_dieldrin_velocity,dt_CS_dieldrin_velocity$id)
for(i in 1:length(split.df)){
  write.csv(split.df[[i]], paste0("/home/hjones/CS/dieldrin/CS_dieldrin_",i,
                                  ".csv"))
}

In [ ]:
#to remove all empty files in a folder - go through each folder and remove empty files
## Get vector of all file names
ff <- dir("/home/hjones/CS/solvent", recursive=TRUE, full.names=TRUE)
## Extract vector of empty files' names
eff <- ff[file.info(ff)[["size"]]==23]
## Remove empty files
unlink(eff, recursive=TRUE, force=FALSE)